In [16]:
import os
import json
import requests

import pandas as pd
import dotenv
import redis
import numpy as np

In [74]:
# Convert data from byte into datatpyes
def convert_from_byte(byte_dict):
    return {key.decode('utf-8'): value.decode('utf-8') for key, value in byte_dict.items()}

In [120]:
# open .env file and get API keys
env_path = os.path.abspath('../.env.development.local')
dotenv.load_dotenv(env_path)
KV_REST_API_READ_ONLY_TOKEN = os.getenv("KV_REST_API_READ_ONLY_TOKEN")
KV_REST_API_TOKEN = os.getenv("KV_REST_API_TOKEN")
KV_REST_API_URL = os.getenv("KV_REST_API_URL")
KV_URL = os.getenv("KV_URL")

# Set headers for authentication
headers = {
    "Authorization": f"Bearer {KV_REST_API_TOKEN}",
    "Content-Type": "application/json"
}

In [121]:
# Adjust url to work with redis
redis_url = KV_URL
if redis_url.startswith("redis://"):
    redis_url = 'rediss://' + redis_url[len('redis://'):]
r = redis.from_url(redis_url)

In [ ]:
# Import datasets
# datasets = {}
# dataset_names = ['clfever', 'phemeplus', 'vitc']
# for dataset_name in dataset_names:
#     with open(f'{dataset_name}.json') as f:
#         datasets[dataset_name] = json.load(f)

In [28]:
# Import VITC daraset
with open("vitc_evaluation_sup_ref.json") as f:
    vitc = json.load(f)

# Randomise order
np.random.shuffle(vitc)
# Test labels
labels = []
for claim in vitc:
    labels.append(claim['label'])
labels = np.array(labels)
np.unique(labels, return_counts=True)

(array(['REFUTES', 'SUPPORTS'], dtype='<U8'), array([217, 283]))

In [50]:
# Populate Vercel KV with vitc datasets
for datapoint in vitc:
    id = datapoint['claim_id']
    r.hset(id, mapping={
        'claim': datapoint['claim'],
        'evidence': datapoint['evidence'],
        'label': datapoint['label']
    })

In [52]:
vitc_ids = [datapoint['claim_id'] for datapoint in vitc]

In [55]:
# Create three batches for VITC
# 100 samples are included in all batches to test for inter-rater reliability

repeated_samples = vitc_ids[:100]

vitc_batches = {
    'vitc_repeated': repeated_samples 
}
start_index = 100
for i in range(3):
    end_index = start_index + ((len(vitc_ids) - 100) // 3) if i < 2 else len(vitc_ids)
    print(f'{start_index} - {end_index}')
    unique_samples = vitc_ids[start_index:end_index]
    start_index = end_index
    vitc_batches[f'vitc{i+1}_workpackage1'] = unique_samples[:15]
    vitc_batches[f'vitc{i+1}_workpackage3'] = unique_samples[15:]

for key in vitc_batches:
    print(key, len(vitc_batches[key])) 

100 - 233
233 - 366
366 - 500
vitc_repeated 100
vitc1_workpackage1 15
vitc1_workpackage3 118
vitc2_workpackage1 15
vitc2_workpackage3 118
vitc3_workpackage1 15
vitc3_workpackage3 119


In [58]:
# Populate vercel KV with VITC batches
for batch_id in vitc_batches.keys():
    claim_ids = vitc_batches[batch_id]
    r.hset(batch_id, mapping={
        'claim_ids': json.dumps(claim_ids)
    })

In [155]:
# Assign batches to annotators
vitc_annotators = {
    'test': 'vitc1',
    'mahmud': 'vitc2',
}

In [ ]:
# r.delete("test")

1

In [156]:
# Upload annotators to Vercel KV
for annotator_id in vitc_annotators.keys():
    r.hset(annotator_id, mapping={
        'stage': 'workpackage1',
        'stage_to_batch': json.dumps({
            'workpackage1': f'{vitc_annotators[annotator_id]}_workpackage1',
            'workpackage2': 'vitc_repeated',
            'workpackage3': f'{vitc_annotators[annotator_id]}_workpackage3'
        }),
        'workpackage1_progress': 0,
        'workpackage2_progress': 0,
        'workpackage3_progress': 0
    })

In [114]:
r.hset('test', 'stage', 'workpackage2')
r.hset('mahmud', 'stage', 'workpackage2')

0

In [119]:
test_data = r.hgetall('mahmud')
test_data = convert_from_byte(test_data)
test_data

{'vitc_16163': '["deductive","Evidence explicitly mentions DNA to be poetry audiobook. "]',
 'vitc_4172': '["deductive","direct evidence"]',
 'vitc_15718': '["deductive","Explicit evidence. Third largest city in Spain. "]',
 'vitc_9219': '["deductive","Direct evidence. "]',
 'stage_to_batch': '{"workpackage1": "vitc2_workpackage1", "workpackage2": "vitc_repeated", "workpackage3": "vitc2_workpackage3"}',
 'vitc_1017': '["deductive","Direct evidence"]',
 'stage': 'workpackage2',
 'workpackage3_progress': '0',
 'vitc_13414': '["deductive","Direct evidence"]',
 'vitc_13841': '["deductive","direct evidence"]',
 'workpackage2_progress': '0',
 'vitc_11539': '["deductive","Direct evidence"]',
 'vitc_1038': '["abductive","The evidence talks so far that means this is ongoing whereas the claim is in the past. Therefore, it is likely Alabama scored more than 9 games over the full season. "]',
 'vitc_925': '["deductive","direct evidence"]',
 'vitc_15978': '["deductive","direct evidence"]',
 'vitc_1

In [15]:
# Get list of all submissions from Prolific (when id wasn't created prior to study)
r.lrange('participants',0,-1)

[]

In [14]:
r.hgetall("66ef9ea77118ba546f2c7c50")

{}

In [274]:
# get ids of participants who completed the task
completed_batches = []
completed_participants = []
submissions = r.lrange('participants',0,-26)
for submission in submissions:
    # convert to dictionary from bytes
    submission = submission.decode('utf-8')
    submission = json.loads(submission)

    if submission["stage"] == "annotation":
        completed_batches.append(submission["batchId"])
        completed_participants.append(submission["participant"])

In [280]:
len(completed_batches)

15

In [275]:
np.unique(completed_batches, return_counts=True)

(array(['batch_vitc_10', 'batch_vitc_11', 'batch_vitc_12', 'batch_vitc_14',
        'batch_vitc_16', 'batch_vitc_17', 'batch_vitc_18', 'batch_vitc_19',
        'batch_vitc_2', 'batch_vitc_3', 'batch_vitc_4', 'batch_vitc_5',
        'batch_vitc_6', 'batch_vitc_8', 'batch_vitc_9'], dtype='<U13'),
 array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]))

In [287]:
vitc_queue = [id for id in vitc_batches.keys() if id not in completed_batches]
vitc_queue

['batch_vitc_1',
 'batch_vitc_7',
 'batch_vitc_13',
 'batch_vitc_15',
 'batch_vitc_20']

In [289]:
# delete queue and then only add remaining batches
r.delete('queue')
r.lpush('queue', *vitc_queue)

5

In [9]:
r.rpush('queue', *["batch_vitc_1", "batch_vitc_2", "batch_vitc_3", "batch_vitc_4"])

20

In [11]:
queue = r.lrange('queue', 0, -1)
print(len(queue))
print(queue)

18
[b'batch_vitc_3', b'batch_vitc_4', b'batch_vitc_1', b'batch_vitc_2', b'batch_vitc_3', b'batch_vitc_4', b'batch_vitc_1', b'batch_vitc_2', b'batch_vitc_3', b'batch_vitc_4', b'batch_vitc_1', b'batch_vitc_2', b'batch_vitc_3', b'batch_vitc_4', b'batch_vitc_1', b'batch_vitc_2', b'batch_vitc_3', b'batch_vitc_4']


In [277]:
dataset = []
for participant in completed_participants:
    submission = r.hgetall(participant)
    submission = convert_from_byte(submission)
    for key in submission:
        if "asses" not in key:
            data = vitc_dict[key]
            data['reasoning'] = submission[key]
            data['participant'] = participant
            dataset.append(data)


In [278]:
dataset = pd.DataFrame(dataset)
dataset.to_json('dataset.json', orient='records', lines=True)

In [279]:
dataset['reasoning'].value_counts()

reasoning
deductive    231
abductive    144
Name: count, dtype: int64

## Phemplus

In [150]:
phemeplus_annotators = {
    'bleiz': 'phemeplus1',
    'nelly': 'phemeplus2',
    'yazhou': 'phemeplus3'
}

In [135]:


# Import phemeplus dataset
with open("phemeplus_incomplete_15-11-24.json") as f:
    phemeplus = json.load(f)

# Randomise order
np.random.shuffle(phemeplus)
# Test labels
labels = []
for claim in phemeplus:
    labels.append(claim['label'])
labels = np.array(labels)
np.unique(labels, return_counts=True)

(array([False,  True]), array([ 91, 202]))

In [124]:
# Populate Vercel KV with phemeplus datasets
for datapoint in phemeplus:
    id = datapoint['claim_id']
    r.hset(id, mapping={
        'claim': datapoint['claim'],
        'evidence': datapoint['evidence'],
        'label': "true" if datapoint['label'] else "false"
    })

In [138]:
phemeplus_ids = [datapoint['claim_id'] for datapoint in phemeplus]

In [145]:
# Create three batches for PHEMEPLUS workpackage 1 (phemeplus is incomplete so far)
repeated_samples = phemeplus_ids[:100]
workpackage1_samples1 = phemeplus_ids[100:115]
workpackage1_samples2 = phemeplus_ids[115:130]
workpackage1_samples3 = phemeplus_ids[130:145]

phemeplus_batches = {
    'phemeplus_repeated': repeated_samples,
    'phemeplus1_workpackage1': workpackage1_samples1,
    'phemeplus2_workpackage1': workpackage1_samples2,
    'phemeplus3_workpackage1': workpackage1_samples3
}

for key in phemeplus_batches:
    print(key, len(phemeplus_batches[key]))


phemeplus_repeated 100
phemeplus1_workpackage1 15
phemeplus2_workpackage1 15
phemeplus3_workpackage1 15


In [147]:
# Populate vercel KV with phemeplus batches
for batch_id in phemeplus_batches.keys():
    claim_ids = phemeplus_batches[batch_id]
    r.hset(batch_id, mapping={
        'claim_ids': json.dumps(claim_ids)
    })

In [151]:
# Upload annotators to Vercel KV
for annotator_id in phemeplus_annotators.keys():
    r.hset(annotator_id, mapping={
        'stage': 'workpackage1',
        'stage_to_batch': json.dumps({
            'workpackage1': f'{phemeplus_annotators[annotator_id]}_workpackage1',
            'workpackage2': 'phemeplus_repeated',
            # 'workpackage3': f'{vitc_annotators[annotator_id]}_workpackage3'
        }),
        'workpackage1_progress': 0,
        'workpackage2_progress': 0,
        'workpackage3_progress': 0
    })

In [154]:
r.hgetall('yazhou')

{b'stage': b'workpackage1',
 b'stage_to_batch': b'{"workpackage1": "phemeplus3_workpackage1", "workpackage2": "phemeplus_repeated"}',
 b'workpackage1_progress': b'0',
 b'workpackage2_progress': b'0',
 b'workpackage3_progress': b'0'}